# Forecasting pipeline v2-v5

Notebook nay duoc dong bo tu `train_forecasting_v2.py`. Pipeline train den het 2021, validate tren 2022, them calendar blend, validation calibration, residual meta-model va LightGBM residual meta-model de tao submission 2023-2024.

## 1. Setup

In [ ]:
from __future__ import annotations

import argparse
import json
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from lightgbm import LGBMRegressor
except Exception:  # pragma: no cover - handled at runtime
    LGBMRegressor = None


warnings.filterwarnings("ignore")

RANDOM_SEED = 42
TARGETS = ["Revenue", "COGS"]
TRAIN_END_FOR_VALID = pd.Timestamp("2021-12-31")
VALID_START = pd.Timestamp("2022-01-01")
VALID_END = pd.Timestamp("2022-12-31")

LAGS = [1, 2, 3, 7, 14, 28, 56, 91, 182, 365]
ROLL_WINDOWS = [7, 14, 28, 56, 91, 182]


## 2. Data loading va feature lich

In [ ]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents, Path(r"D:\DataThon")]:
        if (root / "dataset" / "sales.csv").exists() and (
            root / "dataset" / "sample_submission.csv"
        ).exists():
            return root
    raise FileNotFoundError("Cannot find dataset/sales.csv and dataset/sample_submission.csv")

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out["Date"])
    out["year"] = dt.dt.year.astype(int)
    out["quarter"] = dt.dt.quarter.astype(int)
    out["month"] = dt.dt.month.astype(int)
    out["day"] = dt.dt.day.astype(int)
    out["doy"] = dt.dt.dayofyear.astype(int)
    out["dow"] = dt.dt.dayofweek.astype(int)
    out["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    out["is_weekend"] = (out["dow"] >= 5).astype(int)
    out["is_month_start"] = dt.dt.is_month_start.astype(int)
    out["is_month_end"] = dt.dt.is_month_end.astype(int)
    out["is_quarter_start"] = dt.dt.is_quarter_start.astype(int)
    out["is_quarter_end"] = dt.dt.is_quarter_end.astype(int)
    out["is_year_start"] = dt.dt.is_year_start.astype(int)
    out["is_year_end"] = dt.dt.is_year_end.astype(int)
    out["time_idx"] = (dt - pd.Timestamp("2012-07-04")).dt.days.astype(int)

    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7.0)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12.0)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12.0)
    out["doy_sin"] = np.sin(2 * np.pi * out["doy"] / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * out["doy"] / 365.25)
    return out

def load_inputs(project_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    sales = (
        pd.read_csv(project_root / "dataset" / "sales.csv", parse_dates=["Date"])
        .sort_values("Date")
        .reset_index(drop=True)
    )
    template = (
        pd.read_csv(project_root / "dataset" / "sample_submission.csv", parse_dates=["Date"])[
            ["Date", "Revenue", "COGS"]
        ]
        .sort_values("Date")
        .reset_index(drop=True)
    )

    expected_cols = ["Date", "Revenue", "COGS"]
    if list(sales.columns) != expected_cols:
        raise ValueError(f"sales.csv columns must be {expected_cols}, got {list(sales.columns)}")
    if list(template.columns) != expected_cols:
        raise ValueError(
            f"sample_submission.csv columns must be {expected_cols}, got {list(template.columns)}"
        )
    if not sales["Date"].is_monotonic_increasing:
        raise ValueError("sales.csv dates must be sorted")
    if sales[TARGETS].isna().any().any():
        raise ValueError("sales.csv contains missing targets")
    if (sales[TARGETS] < 0).any().any():
        raise ValueError("sales.csv contains negative targets")
    return sales, template


## 3. Metric va seasonal baseline

In [ ]:
def evaluate_predictions(actual: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    return {
        "MAE": float(mean_absolute_error(actual, pred)),
        "RMSE": float(mean_squared_error(actual, pred) ** 0.5),
        "R2": float(r2_score(actual, pred)),
    }

class SeasonalTrendForecaster:
    def __init__(self, seasonal_years: int, trend_years: int):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)

    def fit(self, train: pd.DataFrame) -> "SeasonalTrendForecaster":
        self.profile_md_: dict[str, pd.DataFrame] = {}
        self.profile_doy_: dict[str, pd.DataFrame] = {}
        self.last_level_: dict[str, float] = {}
        self.last_year_: dict[str, int] = {}
        self.trend_rate_: dict[str, float] = {}

        hist = add_calendar_features(train[["Date"] + TARGETS].copy())
        full_years = hist.groupby("year").filter(lambda x: len(x) >= 360).copy()
        if full_years.empty:
            raise ValueError("Need at least one complete year to fit seasonal profile")

        for target in TARGETS:
            annual = full_years.groupby("year")[target].mean().astype(float)
            self.last_year_[target] = int(annual.index.max())
            self.last_level_[target] = float(annual.iloc[-1])

            keep_years = annual.index[-min(self.seasonal_years, len(annual)) :]
            src = full_years[full_years["year"].isin(keep_years)].copy()
            src["seasonal_index"] = src[target] / src.groupby("year")[target].transform("mean")

            self.profile_md_[target] = (
                src.groupby(["month", "day"])["seasonal_index"].mean().reset_index()
            )
            self.profile_doy_[target] = (
                src.groupby("doy")["seasonal_index"]
                .mean()
                .reset_index(name="seasonal_index_doy")
            )

            if self.trend_years <= 0:
                self.trend_rate_[target] = 1.0
                continue

            trend_source = annual.tail(min(self.trend_years + 1, len(annual)))
            yoy = trend_source.pct_change().dropna()
            if len(yoy) == 0:
                self.trend_rate_[target] = 1.0
            else:
                self.trend_rate_[target] = float(np.exp(np.log1p(yoy).mean()))
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        future = add_calendar_features(
            pd.DataFrame({"Date": pd.to_datetime(dates)}).reset_index(drop=True)
        )
        out = future[["Date"]].copy()

        for target in TARGETS:
            tmp = future.merge(self.profile_md_[target], on=["month", "day"], how="left")
            tmp = tmp.merge(self.profile_doy_[target], on="doy", how="left")
            seasonal_index = (
                tmp["seasonal_index"].fillna(tmp["seasonal_index_doy"]).fillna(1.0).to_numpy(float)
            )
            years_ahead = tmp["year"].to_numpy(float) - self.last_year_[target]
            level = self.last_level_[target] * np.power(self.trend_rate_[target], years_ahead)
            out[target] = np.maximum(level * seasonal_index, 0.0)
        return out


## 4. LightGBM recursive model

In [ ]:
def add_lag_features(df: pd.DataFrame, target: str) -> pd.DataFrame:
    out = add_calendar_features(df[["Date", target]].copy())
    shifted = out[target].shift(1)

    for lag in LAGS:
        out[f"{target}_lag_{lag}"] = out[target].shift(lag)

    for window in ROLL_WINDOWS:
        roll = shifted.rolling(window=window, min_periods=window)
        out[f"{target}_roll_mean_{window}"] = roll.mean()
        out[f"{target}_roll_std_{window}"] = roll.std()
        out[f"{target}_roll_min_{window}"] = roll.min()
        out[f"{target}_roll_max_{window}"] = roll.max()

    out[f"{target}_ewm_7"] = shifted.ewm(span=7, adjust=False, min_periods=7).mean()
    out[f"{target}_ewm_28"] = shifted.ewm(span=28, adjust=False, min_periods=28).mean()
    out[f"{target}_ewm_91"] = shifted.ewm(span=91, adjust=False, min_periods=91).mean()
    return out

@dataclass
class LGBMConfig:
    name: str
    params: dict[str, Any]

class RecursiveLGBMForecaster:
    def __init__(self, config: LGBMConfig):
        if LGBMRegressor is None:
            raise RuntimeError("lightgbm is not installed in this environment")
        self.config = config

    def fit(self, train: pd.DataFrame) -> "RecursiveLGBMForecaster":
        self.history_ = train[["Date"] + TARGETS].sort_values("Date").reset_index(drop=True)
        self.models_: dict[str, Any] = {}
        self.feature_columns_: dict[str, list[str]] = {}

        for target in TARGETS:
            frame = add_lag_features(self.history_[["Date", target]], target)
            feature_cols = [c for c in frame.columns if c not in ["Date", target]]
            fit_frame = frame.dropna(subset=feature_cols + [target]).copy()
            if len(fit_frame) < 200:
                raise ValueError(f"Not enough rows to train target {target}")

            model = LGBMRegressor(
                objective="regression",
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbosity=-1,
                **self.config.params,
            )
            model.fit(fit_frame[feature_cols], np.log1p(fit_frame[target].to_numpy(float)))
            self.models_[target] = model
            self.feature_columns_[target] = feature_cols
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        dates = pd.to_datetime(pd.Series(dates)).sort_values().reset_index(drop=True)
        history = self.history_.copy()
        rows = []

        for date in dates:
            row = {"Date": pd.Timestamp(date)}
            for target in TARGETS:
                tmp = pd.concat(
                    [
                        history[["Date", target]],
                        pd.DataFrame({"Date": [pd.Timestamp(date)], target: [np.nan]}),
                    ],
                    ignore_index=True,
                )
                feature_frame = add_lag_features(tmp, target)
                feature_cols = self.feature_columns_[target]
                x = feature_frame.loc[[len(feature_frame) - 1], feature_cols]
                pred = np.expm1(self.models_[target].predict(x)[0])
                row[target] = float(max(pred, 0.0))
            rows.append(row)
            history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)

        return pd.DataFrame(rows)


## 5. Blend, week/dow seasonal, target scaling, residual meta-model

In [ ]:
class FixedBlendForecaster:
    def __init__(
        self,
        seasonal_years: int,
        trend_years: int,
        lgbm_config: LGBMConfig,
        lgbm_weight: float,
    ):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)
        self.lgbm_config = lgbm_config
        self.lgbm_weight = float(lgbm_weight)

    def fit(self, train: pd.DataFrame) -> "FixedBlendForecaster":
        self.seasonal_model_ = SeasonalTrendForecaster(
            self.seasonal_years, self.trend_years
        ).fit(train)
        self.lgbm_model_ = RecursiveLGBMForecaster(self.lgbm_config).fit(train)
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        seasonal = self.seasonal_model_.predict(dates)
        lgbm = self.lgbm_model_.predict(dates)
        out = seasonal[["Date"]].copy()
        for target in TARGETS:
            out[target] = (
                (1.0 - self.lgbm_weight) * seasonal[target]
                + self.lgbm_weight * lgbm[target]
            )
        return out

class WeekDowSeasonalForecaster:
    def __init__(self, seasonal_years: int, trend_years: int = 0):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)

    def fit(self, train: pd.DataFrame) -> "WeekDowSeasonalForecaster":
        self.profile_week_dow_: dict[str, pd.DataFrame] = {}
        self.profile_md_: dict[str, pd.DataFrame] = {}
        self.profile_dow_: dict[str, pd.DataFrame] = {}
        self.last_level_: dict[str, float] = {}
        self.last_year_: dict[str, int] = {}
        self.trend_rate_: dict[str, float] = {}

        hist = add_calendar_features(train[["Date"] + TARGETS].copy())
        full_years = hist.groupby("year").filter(lambda x: len(x) >= 360).copy()
        if full_years.empty:
            raise ValueError("Need at least one complete year to fit week/dow seasonal profile")

        for target in TARGETS:
            annual = full_years.groupby("year")[target].mean().astype(float)
            self.last_year_[target] = int(annual.index.max())
            self.last_level_[target] = float(annual.iloc[-1])

            keep_years = annual.index[-min(self.seasonal_years, len(annual)) :]
            src = full_years[full_years["year"].isin(keep_years)].copy()
            src["seasonal_index"] = src[target] / src.groupby("year")[target].transform("mean")

            self.profile_week_dow_[target] = (
                src.groupby(["weekofyear", "dow"])["seasonal_index"].mean().reset_index()
            )
            self.profile_md_[target] = (
                src.groupby(["month", "day"])["seasonal_index"].mean().reset_index(name="md_index")
            )
            self.profile_dow_[target] = (
                src.groupby("dow")["seasonal_index"].mean().reset_index(name="dow_index")
            )

            if self.trend_years <= 0:
                self.trend_rate_[target] = 1.0
            else:
                trend_source = annual.tail(min(self.trend_years + 1, len(annual)))
                yoy = trend_source.pct_change().dropna()
                self.trend_rate_[target] = float(np.exp(np.log1p(yoy).mean())) if len(yoy) else 1.0
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        future = add_calendar_features(
            pd.DataFrame({"Date": pd.to_datetime(dates)}).reset_index(drop=True)
        )
        out = future[["Date"]].copy()

        for target in TARGETS:
            tmp = future.merge(
                self.profile_week_dow_[target], on=["weekofyear", "dow"], how="left"
            )
            tmp = tmp.merge(self.profile_md_[target], on=["month", "day"], how="left")
            tmp = tmp.merge(self.profile_dow_[target], on="dow", how="left")
            seasonal_index = (
                tmp["seasonal_index"].fillna(tmp["md_index"]).fillna(tmp["dow_index"]).fillna(1.0)
            ).to_numpy(float)
            years_ahead = tmp["year"].to_numpy(float) - self.last_year_[target]
            level = self.last_level_[target] * np.power(self.trend_rate_[target], years_ahead)
            out[target] = np.maximum(level * seasonal_index, 0.0)
        return out

class CalendarSeasonalBlendForecaster:
    def __init__(self, seasonal_years: int, trend_years: int, week_weight: float):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)
        self.week_weight = float(week_weight)

    def fit(self, train: pd.DataFrame) -> "CalendarSeasonalBlendForecaster":
        self.month_day_model_ = SeasonalTrendForecaster(
            self.seasonal_years, self.trend_years
        ).fit(train)
        self.week_dow_model_ = WeekDowSeasonalForecaster(
            self.seasonal_years, self.trend_years
        ).fit(train)
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        month_day = self.month_day_model_.predict(dates)
        week_dow = self.week_dow_model_.predict(dates)
        out = month_day[["Date"]].copy()
        for target in TARGETS:
            out[target] = (
                (1.0 - self.week_weight) * month_day[target]
                + self.week_weight * week_dow[target]
            )
        return out

class TargetScaledForecaster:
    def __init__(self, base_forecaster: Any, target_scales: dict[str, float]):
        self.base_forecaster = base_forecaster
        self.target_scales = {target: float(target_scales[target]) for target in TARGETS}

    def fit(self, train: pd.DataFrame) -> "TargetScaledForecaster":
        self.base_forecaster.fit(train)
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        out = self.base_forecaster.predict(dates)
        for target in TARGETS:
            out[target] = out[target] * self.target_scales[target]
        return out

class ResidualMetaForecaster:
    """Learn a residual correction from the latest complete year.

    The base models learn the daily shape. The meta model learns how much each
    base forecast should influence the final residual correction.
    """

    def __init__(
        self,
        calibration_year: int | None = None,
        meta_model: str = "huber",
        feature_set: str = "base5",
        correction_shrinkage: float = 1.0,
    ):
        self.calibration_year = calibration_year
        self.meta_model = meta_model
        self.feature_set = feature_set
        self.correction_shrinkage = float(correction_shrinkage)
        self.base_model_factories = {
            "md4": lambda: SeasonalTrendForecaster(4, 0),
            "md5": lambda: SeasonalTrendForecaster(5, 0),
            "md6": lambda: SeasonalTrendForecaster(6, 0),
            "md7": lambda: SeasonalTrendForecaster(7, 0),
            "md8": lambda: SeasonalTrendForecaster(8, 0),
            "wd6": lambda: WeekDowSeasonalForecaster(6, 0),
            "wd7": lambda: WeekDowSeasonalForecaster(7, 0),
            "calblend6w25": lambda: CalendarSeasonalBlendForecaster(6, 0, 0.25),
            "calblend6w50": lambda: CalendarSeasonalBlendForecaster(6, 0, 0.50),
            "calblend7w25": lambda: CalendarSeasonalBlendForecaster(7, 0, 0.25),
        }
        self.residual_base_name = "calblend6w25"
        self.calendar_feature_columns = [
            "month",
            "day",
            "doy",
            "dow",
            "weekofyear",
            "is_weekend",
            "is_month_start",
            "is_month_end",
            "time_idx",
            "dow_sin",
            "dow_cos",
            "month_sin",
            "month_cos",
            "doy_sin",
            "doy_cos",
        ]

    def _select_calibration_year(self, train: pd.DataFrame) -> int:
        hist = add_calendar_features(train[["Date"] + TARGETS].copy())
        full_year_counts = hist.groupby("year").size()
        full_years = full_year_counts[full_year_counts >= 360].index.tolist()
        if not full_years:
            raise ValueError("Need at least one complete year for residual meta calibration")
        if self.calibration_year is not None:
            if self.calibration_year not in full_years:
                raise ValueError(f"Calibration year {self.calibration_year} is not complete")
            return int(self.calibration_year)
        return int(max(full_years))

    def _fit_base_models(self, train: pd.DataFrame) -> dict[str, Any]:
        return {name: factory().fit(train) for name, factory in self.base_model_factories.items()}

    def _base_feature_frame(
        self,
        dates: pd.Series | list[pd.Timestamp],
        fitted_models: dict[str, Any],
    ) -> pd.DataFrame:
        dates = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
        out = add_calendar_features(pd.DataFrame({"Date": dates}))

        for name, model in fitted_models.items():
            pred = model.predict(dates)
            for target in TARGETS:
                out[f"{name}_{target}"] = pred[target].to_numpy(float)

        for target in TARGETS:
            base_cols = [f"{name}_{target}" for name in self.base_model_factories]
            out[f"base_mean_{target}"] = out[base_cols].mean(axis=1)
            out[f"base_median_{target}"] = out[base_cols].median(axis=1)
            out[f"base_std_{target}"] = out[base_cols].std(axis=1)
            out[f"base_min_{target}"] = out[base_cols].min(axis=1)
            out[f"base_max_{target}"] = out[base_cols].max(axis=1)
        return out

    def _feature_columns(self, target: str) -> list[str]:
        if self.feature_set == "base5":
            names = ["md6", "wd6", "calblend6w25", "md7", "calblend7w25"]
            return [f"{name}_{target}" for name in names]

        if self.feature_set == "base5_cal":
            names = ["md6", "wd6", "calblend6w25", "md7", "calblend7w25"]
            return [f"{name}_{target}" for name in names] + self.calendar_feature_columns

        if self.feature_set == "base_all_cal":
            names = list(self.base_model_factories) + [
                "base_mean",
                "base_median",
                "base_std",
                "base_min",
                "base_max",
            ]
            return [f"{name}_{target}" for name in names] + self.calendar_feature_columns

        raise ValueError(f"Unknown residual meta feature_set: {self.feature_set}")

    def _make_meta_model(self) -> Any:
        if self.meta_model == "huber":
            return make_pipeline(
                StandardScaler(),
                HuberRegressor(alpha=0.001, epsilon=1.35, max_iter=1000),
            )

        if self.meta_model == "lgbm":
            if LGBMRegressor is None:
                raise RuntimeError("lightgbm is not installed in this environment")
            return LGBMRegressor(
                objective="mae",
                n_estimators=120,
                learning_rate=0.03,
                num_leaves=7,
                min_child_samples=20,
                reg_lambda=10.0,
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbosity=-1,
            )

        raise ValueError(f"Unknown residual meta_model: {self.meta_model}")

    def fit(self, train: pd.DataFrame) -> "ResidualMetaForecaster":
        train = train[["Date"] + TARGETS].sort_values("Date").reset_index(drop=True)
        self.calibration_year_ = self._select_calibration_year(train)
        cal_start = pd.Timestamp(f"{self.calibration_year_}-01-01")
        cal_end = pd.Timestamp(f"{self.calibration_year_}-12-31")

        base_train = train[train["Date"] < cal_start].copy()
        calibration = train[(train["Date"] >= cal_start) & (train["Date"] <= cal_end)].copy()
        if base_train.empty or len(calibration) < 360:
            raise ValueError("Not enough data to fit residual meta model")

        cal_base_models = self._fit_base_models(base_train)
        cal_features = self._base_feature_frame(calibration["Date"], cal_base_models)
        cal_features = cal_features.merge(calibration[["Date"] + TARGETS], on="Date")

        self.meta_models_: dict[str, Any] = {}
        self.feature_columns_: dict[str, list[str]] = {}
        self.training_rows_ = len(cal_features)

        calibration_prediction = cal_features[["Date"]].copy()
        for target in TARGETS:
            feature_cols = self._feature_columns(target)
            base_col = f"{self.residual_base_name}_{target}"
            residual = cal_features[target].to_numpy(float) - cal_features[base_col].to_numpy(float)
            model = self._make_meta_model()
            model.fit(cal_features[feature_cols], residual)
            self.meta_models_[target] = model
            self.feature_columns_[target] = feature_cols
            correction = self.correction_shrinkage * model.predict(cal_features[feature_cols])
            calibration_prediction[target] = np.maximum(
                cal_features[base_col].to_numpy(float) + correction,
                0.0,
            )

        self.calibration_prediction_ = calibration_prediction
        self.final_base_models_ = self._fit_base_models(train)
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        features = self._base_feature_frame(dates, self.final_base_models_)
        out = features[["Date"]].copy()
        for target in TARGETS:
            base_col = f"{self.residual_base_name}_{target}"
            correction = self.correction_shrinkage * self.meta_models_[target].predict(
                features[self.feature_columns_[target]]
            )
            out[target] = np.maximum(features[base_col].to_numpy(float) + correction, 0.0)
        return out


## 6. Backtest helpers

In [ ]:
def metric_rows(
    model: str,
    variant: str,
    pred: pd.DataFrame,
    valid: pd.DataFrame,
    extra: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    merged = valid[["Date"] + TARGETS].merge(pred[["Date"] + TARGETS], on="Date", suffixes=("_actual", "_pred"))
    rows = []
    for target in TARGETS:
        metrics = evaluate_predictions(
            merged[f"{target}_actual"].to_numpy(float), merged[f"{target}_pred"].to_numpy(float)
        )
        row = {
            "model": model,
            "variant": variant,
            "target": target,
            "train_start": None,
            "train_end": TRAIN_END_FOR_VALID.date().isoformat(),
            "valid_start": VALID_START.date().isoformat(),
            "valid_end": VALID_END.date().isoformat(),
            **metrics,
        }
        if extra:
            row.update(extra)
        rows.append(row)
    return rows

def optimize_target_scales(
    valid: pd.DataFrame,
    pred: pd.DataFrame,
    low: float = 0.85,
    high: float = 1.25,
    step: float = 0.001,
) -> tuple[dict[str, float], pd.DataFrame]:
    scaled = pred.copy()
    scales: dict[str, float] = {}
    grid = np.arange(low, high + step / 2, step)

    for target in TARGETS:
        actual = valid[target].to_numpy(float)
        base = pred[target].to_numpy(float)
        maes = [mean_absolute_error(actual, base * scale) for scale in grid]
        best_scale = float(grid[int(np.argmin(maes))])
        scales[target] = best_scale
        scaled[target] = base * best_scale
    return scales, scaled

def add_combined_scores(metrics: pd.DataFrame) -> pd.DataFrame:
    summary = (
        metrics.groupby(["model", "variant"], as_index=False)
        .agg(MAE_sum=("MAE", "sum"), RMSE_sum=("RMSE", "sum"), R2_mean=("R2", "mean"))
        .sort_values(["MAE_sum", "RMSE_sum"])
        .reset_index(drop=True)
    )
    summary["rank"] = np.arange(1, len(summary) + 1)
    return metrics.merge(summary, on=["model", "variant"], how="left")

def build_backtest_table(best_pred: pd.DataFrame, valid: pd.DataFrame, model: str, variant: str) -> pd.DataFrame:
    out = valid[["Date"] + TARGETS].merge(
        best_pred[["Date"] + TARGETS], on="Date", suffixes=("_actual", "_pred")
    )
    out.insert(1, "model", model)
    out.insert(2, "variant", variant)
    for target in TARGETS:
        out[f"{target}_error"] = out[f"{target}_pred"] - out[f"{target}_actual"]
        out[f"{target}_abs_error"] = out[f"{target}_error"].abs()
    return out

def choose_best_config(metrics: pd.DataFrame) -> tuple[str, str]:
    summary = (
        metrics.groupby(["model", "variant"], as_index=False)
        .agg(MAE_sum=("MAE", "sum"), RMSE_sum=("RMSE", "sum"))
        .sort_values(["MAE_sum", "RMSE_sum"])
        .reset_index(drop=True)
    )
    return str(summary.loc[0, "model"]), str(summary.loc[0, "variant"])


## 7. Run pipeline

In [ ]:
def run_pipeline(project_root: Path) -> None:
    np.random.seed(RANDOM_SEED)
    sales, template = load_inputs(project_root)
    train_valid = sales[sales["Date"] <= TRAIN_END_FOR_VALID].copy()
    valid = sales[(sales["Date"] >= VALID_START) & (sales["Date"] <= VALID_END)].copy()

    if valid.empty or valid["Date"].min() != VALID_START or valid["Date"].max() != VALID_END:
        raise ValueError("Validation period 2022-01-01 to 2022-12-31 is not complete")

    output_dir = project_root / "submition"
    output_dir.mkdir(exist_ok=True)

    metrics_rows: list[dict[str, Any]] = []
    predictions_by_key: dict[tuple[str, str], pd.DataFrame] = {}
    forecaster_builders: dict[tuple[str, str], Any] = {}
    scale_candidate_keys: list[tuple[str, str]] = []

    seasonal_year_grid = [3, 4, 5, 6, 7, 8, 9, 10]
    trend_year_grid = [0, 2, 3, 5, 8]
    for seasonal_years in seasonal_year_grid:
        for trend_years in trend_year_grid:
            variant = f"seasonal{seasonal_years}_trend{trend_years}"
            model = SeasonalTrendForecaster(seasonal_years, trend_years).fit(train_valid)
            pred = model.predict(valid["Date"])
            key = ("seasonal_profile", variant)
            predictions_by_key[key] = pred
            forecaster_builders[key] = lambda s=seasonal_years, t=trend_years: SeasonalTrendForecaster(s, t)
            if trend_years == 0:
                scale_candidate_keys.append(key)
            metrics_rows.extend(
                metric_rows(
                    "seasonal_profile",
                    variant,
                    pred,
                    valid,
                    {"seasonal_years": seasonal_years, "trend_years": trend_years},
                )
            )

    for seasonal_years in [5, 6, 7, 8]:
        for week_weight in [0.25, 0.50, 0.75]:
            variant = f"seasonal{seasonal_years}_week{int(week_weight * 100):02d}_trend0"
            model = CalendarSeasonalBlendForecaster(seasonal_years, 0, week_weight).fit(
                train_valid
            )
            pred = model.predict(valid["Date"])
            key = ("calendar_blend", variant)
            predictions_by_key[key] = pred
            forecaster_builders[key] = (
                lambda s=seasonal_years, w=week_weight: CalendarSeasonalBlendForecaster(s, 0, w)
            )
            scale_candidate_keys.append(key)
            metrics_rows.extend(
                metric_rows(
                    "calendar_blend",
                    variant,
                    pred,
                    valid,
                    {
                        "seasonal_years": seasonal_years,
                        "trend_years": 0,
                        "week_weight": week_weight,
                    },
                )
            )

    lgbm_configs = [
        LGBMConfig(
            "lgbm_stable",
            {
                "n_estimators": 700,
                "learning_rate": 0.035,
                "num_leaves": 31,
                "min_child_samples": 35,
                "subsample": 0.9,
                "colsample_bytree": 0.9,
                "reg_lambda": 2.0,
            },
        ),
        LGBMConfig(
            "lgbm_smooth",
            {
                "n_estimators": 550,
                "learning_rate": 0.03,
                "num_leaves": 15,
                "min_child_samples": 60,
                "subsample": 0.95,
                "colsample_bytree": 0.85,
                "reg_lambda": 6.0,
            },
        ),
        LGBMConfig(
            "lgbm_flexible",
            {
                "n_estimators": 850,
                "learning_rate": 0.025,
                "num_leaves": 63,
                "min_child_samples": 25,
                "subsample": 0.85,
                "colsample_bytree": 0.95,
                "reg_lambda": 1.0,
            },
        ),
    ]

    if LGBMRegressor is not None:
        for cfg in lgbm_configs:
            model = RecursiveLGBMForecaster(cfg).fit(train_valid)
            pred = model.predict(valid["Date"])
            key = ("recursive_lgbm", cfg.name)
            predictions_by_key[key] = pred
            forecaster_builders[key] = lambda c=cfg: RecursiveLGBMForecaster(c)
            metrics_rows.extend(
                metric_rows("recursive_lgbm", cfg.name, pred, valid, {"lgbm_config": cfg.name})
            )

        metrics_so_far = pd.DataFrame(metrics_rows)
        seasonal_best_key = choose_best_config(
            metrics_so_far[metrics_so_far["model"] == "seasonal_profile"]
        )
        lgbm_best_key = choose_best_config(metrics_so_far[metrics_so_far["model"] == "recursive_lgbm"])
        seasonal_best = predictions_by_key[seasonal_best_key]
        lgbm_best = predictions_by_key[lgbm_best_key]
        seasonal_parts = seasonal_best_key[1].replace("seasonal", "").split("_trend")
        best_seasonal_years = int(seasonal_parts[0])
        best_trend_years = int(seasonal_parts[1])
        best_lgbm_cfg = next(cfg for cfg in lgbm_configs if cfg.name == lgbm_best_key[1])

        for weight in [0.25, 0.40, 0.50, 0.60, 0.75]:
            pred = seasonal_best[["Date"]].copy()
            for target in TARGETS:
                pred[target] = (1.0 - weight) * seasonal_best[target] + weight * lgbm_best[target]
            variant = f"seasonal_lgbm_w{int(weight * 100):02d}"
            key = ("blend", variant)
            predictions_by_key[key] = pred
            forecaster_builders[key] = lambda w=weight: FixedBlendForecaster(
                best_seasonal_years, best_trend_years, best_lgbm_cfg, w
            )
            metrics_rows.extend(
                metric_rows(
                    "blend",
                    variant,
                    pred,
                    valid,
                    {
                        "seasonal_years": best_seasonal_years,
                        "trend_years": best_trend_years,
                        "lgbm_config": best_lgbm_cfg.name,
                        "lgbm_weight": weight,
                    },
                )
            )

    for base_key in list(scale_candidate_keys):
        base_model, base_variant = base_key
        base_pred = predictions_by_key[base_key]
        target_scales, scaled_pred = optimize_target_scales(valid, base_pred)
        variant = f"{base_variant}_scaled"
        key = (f"{base_model}_valid_scaled", variant)
        base_builder = forecaster_builders[base_key]
        predictions_by_key[key] = scaled_pred
        forecaster_builders[key] = (
            lambda builder=base_builder, scales=target_scales: TargetScaledForecaster(
                builder(), scales
            )
        )
        metrics_rows.extend(
            metric_rows(
                key[0],
                variant,
                scaled_pred,
                valid,
                {
                    "calibrated_on_valid": True,
                    "base_model": base_model,
                    "base_variant": base_variant,
                    "scale_source": "2022_validation",
                    "revenue_scale": target_scales["Revenue"],
                    "cogs_scale": target_scales["COGS"],
                },
            )
        )

    residual_meta = ResidualMetaForecaster(calibration_year=2022).fit(sales)
    key = ("residual_meta", "huber_base_forecasts_cal2022")
    predictions_by_key[key] = residual_meta.calibration_prediction_
    forecaster_builders[key] = lambda: ResidualMetaForecaster(calibration_year=2022)
    metrics_rows.extend(
        metric_rows(
            key[0],
            key[1],
            residual_meta.calibration_prediction_,
            valid,
            {
                "calibrated_on_valid": True,
                "meta_model": "HuberRegressor",
                "meta_target": "residual",
                "meta_features": "md6,wd6,calblend6w25,md7,calblend7w25",
                "meta_feature_set": "base5",
                "meta_shrinkage": 1.0,
                "calibration_year": 2022,
                "base_variant": "calblend6w25",
            },
        )
    )

    if LGBMRegressor is not None:
        for feature_set in ["base_all_cal", "base5_cal"]:
            residual_lgbm = ResidualMetaForecaster(
                calibration_year=2022,
                meta_model="lgbm",
                feature_set=feature_set,
                correction_shrinkage=1.0,
            ).fit(sales)
            variant = f"lgbm_{feature_set}_cal2022"
            key = ("residual_meta_lgbm", variant)
            predictions_by_key[key] = residual_lgbm.calibration_prediction_
            forecaster_builders[key] = (
                lambda fs=feature_set: ResidualMetaForecaster(
                    calibration_year=2022,
                    meta_model="lgbm",
                    feature_set=fs,
                    correction_shrinkage=1.0,
                )
            )
            metrics_rows.extend(
                metric_rows(
                    key[0],
                    key[1],
                    residual_lgbm.calibration_prediction_,
                    valid,
                    {
                        "calibrated_on_valid": True,
                        "meta_model": "LightGBMRegressor",
                        "meta_target": "residual",
                        "meta_features": feature_set,
                        "meta_feature_set": feature_set,
                        "meta_shrinkage": 1.0,
                        "calibration_year": 2022,
                        "base_variant": "calblend6w25",
                    },
                )
            )

    metrics = add_combined_scores(pd.DataFrame(metrics_rows))
    metrics["train_start"] = metrics["train_start"].fillna(
        train_valid["Date"].min().date().isoformat()
    )
    metrics = metrics.sort_values(["rank", "target"]).reset_index(drop=True)
    metrics_file = project_root / "model_comparison_valid_v5.csv"
    try:
        metrics.to_csv(metrics_file, index=False)
    except PermissionError:
        metrics_file = project_root / "model_comparison_valid_v5_locked_fallback.csv"
        metrics.to_csv(metrics_file, index=False)

    best_model, best_variant = choose_best_config(metrics)
    best_key = (best_model, best_variant)
    backtest = build_backtest_table(predictions_by_key[best_key], valid, best_model, best_variant)
    backtest_file = project_root / "optimization_backtest_v5.csv"
    try:
        backtest.to_csv(backtest_file, index=False)
    except PermissionError:
        backtest_file = project_root / "optimization_backtest_v5_locked_fallback.csv"
        backtest.to_csv(backtest_file, index=False)

    final_forecaster = forecaster_builders[best_key]().fit(sales)
    pred_test = final_forecaster.predict(template["Date"])
    submission = pred_test[["Date"] + TARGETS].copy()
    submission[TARGETS] = submission[TARGETS].clip(lower=0).round(2)
    submission["Date"] = template["Date"].dt.strftime("%Y-%m-%d").to_numpy()

    expected_dates = template["Date"].dt.strftime("%Y-%m-%d").tolist()
    if submission["Date"].tolist() != expected_dates:
        raise ValueError("Submission date order differs from sample_submission.csv")
    if submission[TARGETS].isna().any().any():
        raise ValueError("Submission contains missing predictions")
    if (submission[TARGETS] < 0).any().any():
        raise ValueError("Submission contains negative predictions")

    out_file = output_dir / "submission_train_forecasting_v5_breakthrough.csv"
    submission.to_csv(out_file, index=False, encoding="utf-8-sig", lineterminator="\r\n")

    top_validation_rows = (
        metrics.head(8).astype(object).where(pd.notna(metrics.head(8)), None).to_dict(orient="records")
    )
    summary = {
        "validation_train_range": [
            train_valid["Date"].min().date().isoformat(),
            train_valid["Date"].max().date().isoformat(),
        ],
        "validation_range": [VALID_START.date().isoformat(), VALID_END.date().isoformat()],
        "final_train_range": [
            sales["Date"].min().date().isoformat(),
            sales["Date"].max().date().isoformat(),
        ],
        "best_model": best_model,
        "best_variant": best_variant,
        "metrics_file": str(metrics_file),
        "backtest_file": str(backtest_file),
        "submission_file": str(out_file),
        "top_validation_rows": top_validation_rows,
    }
    (project_root / "forecast_v5_summary.json").write_text(
        json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8"
    )

    print("Validation split:")
    print(f"  train: {train_valid['Date'].min().date()} -> {train_valid['Date'].max().date()}")
    print(f"  valid: {valid['Date'].min().date()} -> {valid['Date'].max().date()}")
    print(f"Best pipeline: {best_model} / {best_variant}")
    print("Top validation metrics:")
    print(
        metrics[["rank", "model", "variant", "target", "MAE", "RMSE", "R2"]]
        .head(8)
        .to_string(index=False)
    )
    print(f"Saved metrics: {metrics_file}")
    print(f"Saved backtest: {backtest_file}")
    print(f"Saved submission: {out_file}")


In [ ]:
project_root = find_project_root()
run_pipeline(project_root)
